## House Prices (Data Preparation — Encoding)

Encoding analyses, category orders and plans are migrated from the original Transformations section. This notebook is independent and uses common training data. No encoder is implemented or fitted here yet. Conceptual classification edits below remain local to this review; configs/variable_types.json is not modified.

In [1]:
import pandas as pd
from IPython.display import display

from house_prices_mlops.pipeline import load_stage
from house_prices_mlops.data_tracker import DataTracker

df_before = load_stage("common")
df_hp = df_before.copy()
tracker = DataTracker.load(df_before)
profile = tracker.profile
display(profile)


2026-10-12 02:13:04.309 | INFO     | house_prices_mlops.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/house_prices_cookiecutter/house-prices-mlops


,var,data_type,variable_type,unique_count,missing_count,missing_pct,zero_count,zero_pct,min,max,mean,median,std,skewness,kurtosis
0,Id,int64,identifier,1168,0,0.000000,0.0,0.000000,1.0,1460.0,730.904966,732.5,425.369088,0.000920,-1.215850
1,MSSubClass,string,nominal,15,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MSZoning,object,nominal,5,0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LotFrontage,float64,continuous,107,217,18.578767,0.0,0.000000,21.0,313.0,70.343849,70.0,24.897021,2.406787,19.454319
4,LotArea,int64,continuous,890,0,0.000000,0.0,0.000000,1300.0,215245.0,10689.642123,9600.0,10759.366198,11.958088,186.444082
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,HasBasement,int64,None,2,0,0.000000,28.0,2.397260,0.0,1.0,0.976027,1.0,0.153029,-6.232060,36.901760
89,HasPool,int64,None,2,0,0.000000,1162.0,99.486301,0.0,1.0,0.005137,0.0,0.071519,13.862369,190.491446
90,HasFireplace,int64,None,2,0,0.000000,547.0,46.832192,0.0,1.0,0.531678,1.0,0.499209,-0.127131,-1.987244
91,IsRemodeled,int64,None,2,0,0.000000,601.0,51.455479,0.0,1.0,0.485445,0.0,0.500002,0.058319,-2.000027


### Encoding

In [2]:
# ENCODING

nominal_vars = profile.loc[
    profile['variable_type'].eq('nominal'),
    'var'
].tolist()

ordinal_vars = profile.loc[
    profile['variable_type'].eq('ordinal'),
    'var'
].tolist()


nominal_vars = [
    var for var in nominal_vars
    if var in df_hp.columns
]

ordinal_vars = [
    var for var in ordinal_vars
    if var in df_hp.columns
]

print('Nominal:', len(nominal_vars))
print('Ordinal:', len(ordinal_vars))

Nominal: 24
Ordinal: 22


In [3]:
nominal_analysis = profile.loc[
    profile['var'].isin(nominal_vars),
    [
        'var',
        'unique_count'
    ]
].sort_values(
    'unique_count',
    ascending=False
)

display(nominal_analysis)

,var,unique_count
12,Neighborhood,25
24,Exterior2nd,16
23,Exterior1st,15
1,MSSubClass,15
13,Condition1,9
78,SaleType,9
14,Condition2,8
16,HouseStyle,8
58,GarageType,7
22,RoofMatl,7


In [4]:
for var in nominal_vars:

    tracker.add_plan(
        var=var,
        stage='DP - Encoding',
        action='encode',
        method='onehot',
        params={
            'drop': None,
            'handle_unknown': 'ignore'
        }
    )

#### Ordinal Variables

In [5]:
# ordinal
ordinal_analysis = profile.loc[
    profile['var'].isin(ordinal_vars),
    [
        'var',
        'variable_type',
        'unique_count'
    ]
].copy()

display(ordinal_analysis)

,var,variable_type,unique_count
7,LotShape,ordinal,4
9,Utilities,ordinal,2
11,LandSlope,ordinal,3
17,OverallQual,ordinal,10
18,OverallCond,ordinal,9
27,ExterQual,ordinal,4
28,ExterCond,ordinal,5
30,BsmtQual,ordinal,5
31,BsmtCond,ordinal,5
32,BsmtExposure,ordinal,5


In [6]:
for var in ordinal_vars:
    print(var, df_hp[var].dropna().unique())

LotShape ['Reg' 'IR1' 'IR2' 'IR3']
Utilities ['AllPub' 'NoSeWa']
LandSlope ['Gtl' 'Mod' 'Sev']
OverallQual [ 7  6  8  5  9  4  3 10  1  2]
OverallCond [5 8 6 7 4 2 3 9 1]
ExterQual ['Gd' 'TA' 'Ex' 'Fa']
ExterCond ['TA' 'Gd' 'Fa' 'Po' 'Ex']
BsmtQual ['Gd' 'TA' 'Ex' 'None' 'Fa']
BsmtCond ['TA' 'Gd' 'None' 'Fa' 'Po']
BsmtExposure ['No' 'Gd' 'Mn' 'Av' 'None']
BsmtFinType1 ['GLQ' 'ALQ' 'Unf' 'Rec' 'BLQ' 'None' 'LwQ']
BsmtFinType2 ['Unf' 'BLQ' 'None' 'ALQ' 'Rec' 'LwQ' 'GLQ']
HeatingQC ['Ex' 'Gd' 'TA' 'Fa' 'Po']
KitchenQual ['Gd' 'TA' 'Ex' 'Fa']
Functional ['Typ' 'Min1' 'Maj1' 'Min2' 'Mod' 'Maj2' 'Sev']
FireplaceQu ['None' 'TA' 'Gd' 'Fa' 'Ex' 'Po']
GarageFinish ['RFn' 'Unf' 'Fin' 'None']
GarageQual ['TA' 'Fa' 'Gd' 'None' 'Ex' 'Po']
GarageCond ['TA' 'None' 'Gd' 'Fa' 'Po' 'Ex']
PavedDrive ['Y' 'N' 'P']
PoolQC ['None' 'Ex' 'Fa' 'Gd']
Fence ['None' 'MnPrv' 'GdWo' 'GdPrv' 'MnWw']


In [7]:
ordinal_orders = {

    'LotShape': ['IR3', 'IR2', 'IR1', 'Reg'],

    'Utilities': ['NoSeWa', 'AllPub'],

    'LandSlope': ['Sev', 'Mod', 'Gtl'],

    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],

    'BsmtQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],

    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],

    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],

    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1','Mod', 'Min2', 'Min1', 'Typ'],

    'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],

    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],

    'PavedDrive': ['N', 'P', 'Y'],

    'PoolQC': ['None', 'Fa', 'TA', 'Gd', 'Ex']
}

In [8]:
tracker.findings.loc[tracker.findings['var'].eq('Electrical')]

,key,var,stage,finding,status,resolution
2,electrical_missing,Electrical,DU - Quality - Missing Values,1 valores faltantes (0.09%) en entrenamiento. ...,pending,None


In [9]:
#se corrige el tipo de dato ya que analizando no es de tipo ordinal
profile.loc[
    profile['var'].isin(['Electrical', 'Fence']),
    'variable_type'
] = 'nominal'

display(
    profile.loc[
        profile['var'].isin(['Electrical', 'Fence']),
        ['var', 'data_type', 'variable_type']
    ]
)

,var,data_type,variable_type
42,Electrical,object,nominal
73,Fence,object,nominal


In [10]:
for var, categories in ordinal_orders.items():

    tracker.add_plan(
        var=var,
        stage='DP - Encoding',
        action='encode',
        method='ordinal',
        params={
            'categories': categories
        }
    )

In [11]:
display(
    tracker.plans[
        tracker.plans['stage'].eq('DP - Encoding')
    ]
)

,id,var,stage,action,method,params,status
19,20,MSSubClass,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned
20,21,MSZoning,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned
21,22,Street,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned
22,23,Alley,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned
23,24,LandContour,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned
24,25,LotConfig,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned
25,26,Neighborhood,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned
26,27,Condition1,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned
27,28,Condition2,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned
28,29,BldgType,DP - Encoding,encode,onehot,"{'drop': None, 'handle_unknown': 'ignore'}",planned


In [12]:
nominal_vars = tracker.profile.loc[
    tracker.profile['variable_type'].eq('nominal'),
    'var'
].tolist()

ordinal_vars = tracker.profile.loc[
    tracker.profile['variable_type'].eq('ordinal'),
    'var'
].tolist()

print('Nominal:', len(nominal_vars))
print('Ordinal:', len(ordinal_vars))

Nominal: 25
Ordinal: 21


In [13]:
print('Electrical' in nominal_vars)
print('Fence' in nominal_vars)

print('Electrical' in ordinal_vars)
print('Fence' in ordinal_vars)

True
True
False
False


In [14]:
for var in ['Electrical', 'Fence']:
    tracker.add_plan(
        var=var,
        stage='DP - Encoding',
        action='encode',
        method='onehot',
        params={}
    )

### Binning

In [15]:
# BINNING

numeric_analysis = tracker.profile.loc[
    tracker.profile['variable_type'].isin([
        'continuous',
        'discrete'
    ]),
    [
        'var',
        'variable_type',
        'unique_count',
        'min',
        'max'
    ]
].copy()

display(
    numeric_analysis.sort_values(
        'unique_count',
        ascending=True
    )
)

#Binning → evaluado, no aplicado.

,var,variable_type,unique_count,min,max
48,BsmtHalfBath,discrete,3,0.0,2.0
50,HalfBath,discrete,3,0.0,2.0
49,FullBath,discrete,4,0.0,3.0
47,BsmtFullBath,discrete,4,0.0,3.0
56,Fireplaces,discrete,4,0.0,3.0
52,KitchenAbvGr,discrete,4,0.0,3.0
61,GarageCars,discrete,5,0.0,4.0
71,PoolArea,continuous,7,0.0,738.0
51,BedroomAbvGr,discrete,8,0.0,8.0
54,TotRmsAbvGrd,discrete,12,2.0,14.0


### Save Decisions

In [16]:
tracker.save()
